# Lab 1: Network Measurement and Protocol Analysis
**CMSC395 — Computer Networks**  
Estimated time: 3–4 hours | Points: 100

---

Work through each cell in order. Run a cell with **Shift+Enter** and read the output before moving on.  
Do not import `requests`, `urllib`, or any library that abstracts socket operations for Parts 1 and 3.

> **Submission:** Use **Kernel → Restart Kernel and Run All Cells** before your final push.  
> Commit after completing each part — at least 5 commits total are required.

## Setup — Imports

Run this cell first. It imports everything you will need across all three parts.

In [ ]:
# Setup — run this cell before starting any part
import socket
import struct
import time
import os
import json
import ssl
import heapq
import random
import statistics
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Lab server — update if the instructor provides a different hostname
LAB_SERVER = "lab.yourclass.edu"
LAB_HTTP_PORT = 8080
LAB_HTTPS_PORT = 8443

print("Imports OK")
print(f"Lab server: {LAB_SERVER}")

---
## Part 1: Active Path Measurement with ICMP

You will implement traceroute from scratch using raw sockets and ICMP, then extend it with UDP probe support and structured JSON output.

**Allowed imports for this part:** `socket`, `struct`, `time`, `json`, `os` (already imported above).  
You may use `scapy` only for verification, not for sending probes.

### Cell 1.1 - ICMP Packet Construction and Checksum

Implement `build_icmp_packet(identifier, sequence)` that returns a raw ICMP Echo Request packet as bytes.  
Implement `icmp_checksum(data)` that computes the ICMP checksum over a bytes object.

Test by sending a single probe to the lab server and confirming you receive an Echo Reply.  
Print the response source IP and RTT.

**Hint:** ICMP Echo Request is type 8, code 0. The checksum field is bytes 2–3 of the header. Set it to zero before computing, then insert the result.

In [ ]:
# Cell 1.1 — ICMP packet construction and checksum

def icmp_checksum(data):
    """
    Compute the ICMP checksum over data (bytes).
    Returns a 16-bit integer.
    """
    # Your code here
    pass


def build_icmp_packet(identifier, sequence, payload=b'CMSC395-lab1'):
    """
    Build a raw ICMP Echo Request packet.
    Returns bytes ready to send via a raw socket.
    """
    # Your code here
    pass


# Test: send a single ICMP probe to the lab server (TTL=64, no traceroute)
# Print the responding IP and RTT in ms



### Cell 1.2 — Single-Hop Probe

Implement `send_probe(dest_ip, ttl, identifier, sequence, timeout=2.0)` that:
- Sends one ICMP Echo Request with the given TTL
- Waits up to `timeout` seconds for a response
- Returns `(responding_ip, rtt_ms)` or `(None, None)` if no response arrives

The response may be an ICMP Time Exceeded (type 11) from an intermediate router,  
or an ICMP Echo Reply (type 0) from the destination.

In [ ]:
# Cell 1.2 — Single-hop probe

def send_probe(dest_ip, ttl, identifier, sequence, timeout=2.0):
    """
    Send one ICMP probe with the given TTL.
    Returns (responding_ip, rtt_ms) or (None, None) on timeout.
    """
    # Your code here
    pass


# Test: send probes with TTL=1 and TTL=2 to the lab server
# Print the responding IP and RTT for each



### Cell 1.3 — Full Traceroute with JSON Output

Implement `traceroute(destination, max_hops=30, probes_per_hop=3)` that:
- Resolves `destination` to an IP address
- Sends `probes_per_hop` probes at each TTL from 1 to `max_hops`
- Attempts reverse DNS on each responding IP
- Sets `asymmetric_flag=True` if mean RTT for this hop is lower than the previous hop's mean RTT
- Stops when an Echo Reply is received from the destination
- Returns a dict matching the JSON structure in the lab guide

Run your traceroute against **at least two different destinations** and print the JSON output for each.

In [ ]:
# Cell 1.3 — Full traceroute with JSON output

def reverse_dns(ip):
    """Return hostname for ip, or None if lookup fails."""
    try:
        return socket.gethostbyaddr(ip)[0]
    except Exception:
        return None


def traceroute(destination, max_hops=30, probes_per_hop=3):
    """
    Traceroute to destination.
    Returns a dict with 'destination' and 'hops' keys.
    """
    # Your code here
    pass


# Run against at least two destinations and print JSON output
destinations = [LAB_SERVER, "8.8.8.8"]

results = {}
for dest in destinations:
    print(f"\nTracing route to {dest}...")
    result = traceroute(dest)
    results[dest] = result
    print(json.dumps(result, indent=2))


### Cell 1.4 — UDP Probe Mode

Implement `traceroute_udp(destination, max_hops=30, probes_per_hop=3)` that sends UDP probes
instead of ICMP Echo Requests. Use destination port 33434 (the standard traceroute UDP port).

Routers still respond with ICMP Time Exceeded — you receive responses the same way as in Cell 1.2.
The destination responds with ICMP Port Unreachable (type 3, code 3) instead of Echo Reply.

Run UDP traceroute against the same destinations as Cell 1.3 and compare the paths.

In [ ]:
# Cell 1.4 — UDP probe mode

def traceroute_udp(destination, max_hops=30, probes_per_hop=3, base_port=33434):
    """
    UDP-based traceroute.
    Returns a dict with the same structure as traceroute().
    """
    # Your code here
    pass


# Run UDP traceroute against the same destinations
# Print a side-by-side comparison of ICMP vs UDP hop counts and RTTs



### Reflection 1.A

Examine your JSON output and find a hop where the RTT is noticeably higher than its neighbours. What could cause a single link to contribute a large fraction of end-to-end latency?

Now find a hop where the RTT is *lower* than the previous hop — what does this suggest about routing or measurement?

*(Reference specific hop numbers and RTT values from your output.)*

**Your answer:**

*(Write here)*

### Reflection 1.B

Your reverse DNS lookups reveal hostnames for some hops but not others. What information can you infer from a hostname like `ae-3.r01.chcgil09.us.bb.gin.ntt.net`? What does a hop with no hostname suggest?

**Your answer:**

*(Write here)*

---
## Part 2: Queuing Delay Simulator

Build an event simulator that demonstrates the relationship between link utilization and queuing delay.

### Cell 2.1 — Event-Driven Simulator Core

Implement `simulate_queue(link_rate_bps, mean_packet_size_bytes, utilization, num_packets=12000, warmup=1000, max_queue=None)`.

The function should:
- Derive the arrival rate from `utilization`, `link_rate_bps`, and `mean_packet_size_bytes`
- Use `heapq` as a priority queue of `(event_time, event_type)` tuples
- Process ARRIVAL and DEPARTURE events in time order
- Drop packets when the queue is full (if `max_queue` is set) and count drops
- Discard the first `warmup` packets before collecting statistics
- Return a dict with keys: `mean_delay_ms`, `p95_delay_ms`, `mean_queue_length`, `loss_rate`

**Hint:** Utilization ρ = λ / μ where λ is the arrival rate (packets/sec) and μ is the service rate (packets/sec).  
Service rate μ = link_rate_bps / (mean_packet_size_bytes × 8).

In [ ]:
# Cell 2.1 — Event-driven simulator core

def simulate_queue(link_rate_bps, mean_packet_size_bytes, utilization,
                   num_packets=12000, warmup=1000, max_queue=None):
    """
    Discrete-event M/M/1 (or M/M/1/K) queue simulator.

    Returns dict with mean_delay_ms, p95_delay_ms, mean_queue_length, loss_rate.
    """
    # Your code here
    pass


# Quick sanity check at utilization=0.5
result = simulate_queue(
    link_rate_bps=1_000_000,
    mean_packet_size_bytes=1000,
    utilization=0.5
)
print("Sanity check at ρ=0.5:")
for k, v in result.items():
    print(f"  {k}: {v:.4f}")


### Cell 2.2 — Utilization Sweep and Plot

Run the simulator at each utilization value below. Plot simulated mean delay and the M/M/1 theoretical prediction on the same axes.

M/M/1 theoretical mean waiting time:  
E[W] = ρ / (μ × (1 − ρ))

Label both curves, add axis labels, a title, and a legend.

In [ ]:
# Cell 2.2 — Utilization sweep and plot

LINK_RATE = 1_000_000       # 1 Mbps
MEAN_PKT_SIZE = 1000        # bytes
UTILIZATIONS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]

simulated_delays = []
theoretical_delays = []

mu = LINK_RATE / (MEAN_PKT_SIZE * 8)   # service rate (packets/sec)

for rho in UTILIZATIONS:
    # Run simulation
    result = simulate_queue(LINK_RATE, MEAN_PKT_SIZE, rho)
    simulated_delays.append(result['mean_delay_ms'])

    # Theoretical M/M/1 mean waiting time (ms)
    theoretical_ms = (rho / (mu * (1 - rho))) * 1000
    theoretical_delays.append(theoretical_ms)

    print(f"ρ={rho:.2f}  simulated={result['mean_delay_ms']:.2f}ms  "
          f"theoretical={theoretical_ms:.2f}ms")

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

# Your plotting code here

plt.tight_layout()
plt.show()


### Cell 2.3 — Finite Queue Extension

Re-run the utilization sweep with `max_queue=50`. Plot **packet loss rate** vs utilization.

On a second figure, overlay the mean delay with finite queue against the infinite queue results from Cell 2.2 — what effect does the finite queue have on delay at high utilization, and why?

In [ ]:
# Cell 2.3 — Finite queue extension

MAX_QUEUE = 50

finite_delays = []
loss_rates = []

for rho in UTILIZATIONS:
    result = simulate_queue(LINK_RATE, MEAN_PKT_SIZE, rho, max_queue=MAX_QUEUE)
    finite_delays.append(result['mean_delay_ms'])
    loss_rates.append(result['loss_rate'])
    print(f"ρ={rho:.2f}  loss={result['loss_rate']:.4f}  delay={result['mean_delay_ms']:.2f}ms")

# Figure 1: loss rate vs utilization
fig1, ax1 = plt.subplots(figsize=(9, 4))
# Your plotting code here
plt.tight_layout()
plt.show()

# Figure 2: finite vs infinite queue delay
fig2, ax2 = plt.subplots(figsize=(9, 4))
# Your plotting code here
plt.tight_layout()
plt.show()


### Reflection 2.A

Your simulated curve should closely match the M/M/1 theoretical prediction. Describe how well they match. If there is a discrepancy, what might cause it? What assumptions does M/M/1 make that your simulator also makes?

**Your answer:**

*(Write here)*

### Reflection 2.B

A network link running at 90% utilization is not unusual in practice — ISPs often run links at this level. How do real networks manage this without unacceptable delay? What mechanisms not present in your simple M/M/1 model help in practice?

**Your answer:**

*(Write here)*

---
## Part 3: Raw HTTP/1.0 Client with Latency Decomposition

Measure the time spent in each phase of an HTTP transaction using only Python's `socket` module.  
**Do not import `requests`, `urllib`, or `http.client`.**

### Cell 3.1 — HTTP/1.0 Client with Phase Timing

Implement `http_get(host, port, path)` that:
- Resolves the hostname using `socket.getaddrinfo` and records DNS time
- Opens a TCP connection and records connect time
- Sends a valid HTTP/1.0 GET request
- Records time to first byte of the response body
- Reads the full response and records total transfer time
- Returns a dict with keys: `dns_ms`, `connect_ms`, `ttfb_ms`, `transfer_ms`, `status_code`, `response_size_bytes`

Use `time.perf_counter()` for all timing. Record timestamps as close to system calls as possible.

In [ ]:
# Cell 3.1 — HTTP/1.0 client with phase timing

def http_get(host, port, path):
    """
    Fetch http://host:port/path using a raw socket.
    Returns dict with timing phases and response metadata.
    """
    result = {
        'dns_ms': None,
        'connect_ms': None,
        'ttfb_ms': None,
        'transfer_ms': None,
        'status_code': None,
        'response_size_bytes': None,
    }

    # Your code here

    return result


# Quick test against the /ping endpoint
test = http_get(LAB_SERVER, LAB_HTTP_PORT, '/ping')
print("Single request to /ping:")
for k, v in test.items():
    print(f"  {k}: {v}")


### Cell 3.2 — 10-Run Statistical Summary

Call `http_get` 10 times against each endpoint below. Compute mean, standard deviation, min, and max for each timing phase. Display as a formatted table.

Endpoints to test:
- `/ping` (~20 bytes)
- `/rfc/rfc793.txt` (~90 KB)

In [ ]:
# Cell 3.2 — 10-run statistical summary

ENDPOINTS = [
    ('ping',    '/ping'),
    ('rfc793',  '/rfc/rfc793.txt'),
]

PHASES = ['dns_ms', 'connect_ms', 'ttfb_ms', 'transfer_ms']
N_RUNS = 10

for label, path in ENDPOINTS:
    print(f"\n{'='*60}")
    print(f"Endpoint: {path}  ({N_RUNS} runs)")
    print(f"{'='*60}")

    runs = [http_get(LAB_SERVER, LAB_HTTP_PORT, path) for _ in range(N_RUNS)]

    # Print a table: phase | mean | std | min | max
    # Your code here


### Cell 3.3 — HTTPS Extension

Implement `https_get(host, port, path)` by wrapping your socket with `ssl.create_default_context().wrap_socket()`.  
Add a `tls_handshake_ms` phase (time from connected socket to wrapped socket ready to send).

The lab server uses a self-signed certificate — set `check_hostname=False` and `verify_mode=ssl.CERT_NONE`.

Run 10 requests to `/rfc/rfc793.txt` over HTTPS and compare with your HTTP results from Cell 3.2.

In [ ]:
# Cell 3.3 — HTTPS extension

def https_get(host, port, path):
    """
    Fetch https://host:port/path using a raw socket wrapped with TLS.
    Returns dict with dns_ms, connect_ms, tls_handshake_ms,
    ttfb_ms, transfer_ms, status_code, response_size_bytes.
    """
    # Your code here
    pass


# Run 10 requests over HTTPS
https_runs = [https_get(LAB_SERVER, LAB_HTTPS_PORT, '/rfc/rfc793.txt')
              for _ in range(N_RUNS)]

# Print comparison table: HTTP vs HTTPS for each phase
# Your code here


### Cell 3.4 — Transfer Rate vs Object Size

Fetch the three endpoints below and compute effective throughput (bytes/sec) for each.  
Plot `transfer_ms` vs `response_size_bytes` on one figure and throughput (MB/s) vs object size on a second.

Endpoints:
- `/ping` (~20 bytes)
- `/rfc/rfc793.txt` (~90 KB)
- `/data/1mb` (1 MB)

In [ ]:
# Cell 3.4 — Transfer rate vs object size

SIZE_ENDPOINTS = [
    '/ping',
    '/rfc/rfc793.txt',
    '/data/1mb',
]

size_results = []
for path in SIZE_ENDPOINTS:
    r = http_get(LAB_SERVER, LAB_HTTP_PORT, path)
    throughput_mbps = (r['response_size_bytes'] / (r['transfer_ms'] / 1000)) / 1e6
    size_results.append({
        'path': path,
        'size_bytes': r['response_size_bytes'],
        'transfer_ms': r['transfer_ms'],
        'throughput_mbps': throughput_mbps,
    })
    print(f"{path}: {r['response_size_bytes']} bytes, "
          f"{r['transfer_ms']:.2f}ms, {throughput_mbps:.3f} MB/s")

# Figure 1: transfer_ms vs response_size_bytes
# Figure 2: throughput (MB/s) vs response_size_bytes
# Your plotting code here


### Reflection 3.A

Look at your DNS timing across the 10 runs. Does it vary significantly between runs? What explains the variation (or lack of it)? What would you expect to see if you ran the same test from a different network location?

**Your answer:**

*(Write here)*

### Reflection 3.B

Compare your HTTP and HTTPS timing. How large is the TLS handshake overhead in absolute terms? As a fraction of total request time for a small object? For a large object? What does this tell you about the cost of HTTPS for different use cases?

The lab server uses a self-signed certificate and you disabled hostname verification. Why would this be unacceptable in production?

**Your answer:**

*(Write here)*

### Reflection 3.C

Your transfer rate likely varies with object size. If throughput is lower for small objects, what mechanism explains this? Connect your answer to what you know about TCP slow start and the queuing delay model from Part 2.

**Your answer:**

*(Write here)*

---
## Submission Checklist

Before your final push, complete this checklist. Replace each `[ ]` with `[x]` when done.

- [ ] Cell 1.1: ICMP checksum and packet construction working, single probe test passes
- [ ] Cell 1.2: Single-hop probe handles both Time Exceeded and Echo Reply responses
- [ ] Cell 1.3: Full traceroute produces JSON output for at least 2 destinations
- [ ] Cell 1.4: UDP probe mode implemented and compared against ICMP results
- [ ] Reflections 1.A and 1.B completed with references to specific output values
- [ ] Cell 2.1: Simulator runs without errors, sanity check output looks reasonable
- [ ] Cell 2.2: Utilization sweep plot shows both simulated and theoretical curves, labelled
- [ ] Cell 2.3: Finite queue - loss rate plot and delay comparison plot both included
- [ ] Reflections 2.A and 2.B completed
- [ ] Cell 3.1: HTTP client returns all 6 fields correctly for /ping
- [ ] Cell 3.2: 10-run summary table shown for both endpoints
- [ ] Cell 3.3: HTTPS client working, HTTP vs HTTPS comparison shown
- [ ] Cell 3.4: Both plots included, throughput computed for all 3 object sizes
- [ ] Reflections 3.A, 3.B, and 3.C completed
- [ ] Notebook runs cleanly with Kernel → Restart Kernel and Run All Cells
- [ ] At least 5 commits with descriptive messages in repository history

Final commit message must be exactly: `Lab1 final submission`

---
*CMSC395 Computer Networks - Lab 1*  
*Push this notebook to your repository with the message: `Lab1 final submission`*